In [1]:
# Import packages and load the processed TCGA-BRCA TP53 mutation table

import pandas as pd
import numpy as np

tp53_mutations_df = pd.read_csv(
    "../data/processed/tcga_brca_tp53_mutations.csv"
)

print("TP53 mutation rows:", len(tp53_mutations_df))
print("Unique TP53-mutated patients:", tp53_mutations_df["submitter_id"].nunique())

tp53_mutations_df.head()

TP53 mutation rows: 347
Unique TP53-mutated patients: 332


,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Variant_Classification,Variant_Type,...,hotspot,RNA_Support,RNA_depth,RNA_ref_count,RNA_alt_count,callers,file_id,file_name,submitter_id,tumor_aliquot_id
0,TP53,7157,WUGSC,GRCh38,chr17,7674200,7674200,+,Missense_Mutation,SNP,...,Y,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2,7226e23d-6c48-4a24-9c32-f61f3bde7e06,c107d4de-6001-4027-a7f1-7c03d484ddaf.wxs.aliqu...,TCGA-E2-A1L7,TCGA-E2-A1L7-01A-11D-A142-09
1,TP53,7157,WUGSC,GRCh38,chr17,7673796,7673796,+,Missense_Mutation,SNP,...,Y,Unknown,NaN,NaN,NaN,muse;varscan2,cf171a35-7213-4e87-8237-96e93a8ded86,0b8a9b96-d855-482b-97bd-a45ff5a20468.wxs.aliqu...,TCGA-PL-A8LV,TCGA-PL-A8LV-01A-21D-A41F-09
2,TP53,7157,WUGSC,GRCh38,chr17,7676025,7676033,+,In_Frame_Del,DEL,...,N,Unknown,NaN,NaN,NaN,mutect2;varscan2,c0c20a74-6cb9-4123-816e-a26baca93163,5c132854-804e-40cc-a6db-3252ebd928ef.wxs.aliqu...,TCGA-A2-A0SV,TCGA-A2-A0SV-01A-11D-A099-09
3,TP53,7157,WUGSC,GRCh38,chr17,7673704,7673704,+,Nonsense_Mutation,SNP,...,Y,Unknown,NaN,NaN,NaN,mutect2;varscan2,27f59846-9180-440f-8c64-40b3d0c1565f,d58cfc24-67c9-4624-bb0e-f35674ce8f30.wxs.aliqu...,TCGA-E9-A6HE,TCGA-E9-A6HE-01A-11D-A31U-09
4,TP53,7157,WUGSC,GRCh38,chr17,7674894,7674894,+,Nonsense_Mutation,SNP,...,Y,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2,d900d564-8177-4917-9d30-258931436d38,e0d63c88-2776-4608-bfd5-358a02a681a0.wxs.aliqu...,TCGA-AO-A124,TCGA-AO-A124-01A-11D-A10M-09


In [2]:
# Count TP53 mutation classes across the TCGA-BRCA mutation dataset

tp53_class_counts = (
    tp53_mutations_df["Variant_Classification"]
    .value_counts()
)

tp53_class_counts

Variant_Classification
Missense_Mutation    203
Nonsense_Mutation     49
Frame_Shift_Del       49
Splice_Site           24
Frame_Shift_Ins       15
In_Frame_Del           6
Splice_Region          1
Name: count, dtype: int64

In [3]:
# Calculate the percentage distribution of TP53 mutation classes

tp53_class_summary = (
    tp53_mutations_df["Variant_Classification"]
    .value_counts()
    .rename_axis("mutation_class")
    .reset_index(name="count")
)

tp53_class_summary["percentage"] = (
    tp53_class_summary["count"]
    / tp53_class_summary["count"].sum()
    * 100
).round(1)

tp53_class_summary

,mutation_class,count,percentage
0,Missense_Mutation,203,58.5
1,Nonsense_Mutation,49,14.1
2,Frame_Shift_Del,49,14.1
3,Splice_Site,24,6.9
4,Frame_Shift_Ins,15,4.3
5,In_Frame_Del,6,1.7
6,Splice_Region,1,0.3


In [4]:
# Identify the most recurrent TP53 protein-level mutations in TCGA-BRCA

recurrent_tp53_mutations = (
    tp53_mutations_df["HGVSp_Short"]
    .dropna()
    .value_counts()
    .head(20)
)

recurrent_tp53_mutations

HGVSp_Short
p.R175H          20
p.R273H          12
p.R196*           8
p.R342*           7
p.Y220C           7
p.H193R           6
p.H179R           6
p.R213*           6
p.R248W           6
p.I195T           5
p.C176F           5
p.X225_splice     5
p.X261_splice     4
p.R273C           4
p.R282W           4
p.R209Kfs*6       4
p.C242Afs*5       4
p.L194R           4
p.R273L           3
p.G266E           3
Name: count, dtype: int64

In [5]:
# Identify the most frequently mutated TP53 protein positions

tp53_position_counts = (
    tp53_mutations_df["Protein_position"]
    .dropna()
    .astype(str)
    .str.split("/")
    .str[0]
    .value_counts()
    .head(20)
)

tp53_position_counts

Protein_position
175    20
273    20
342    10
220    10
193     9
248     8
196     8
176     7
179     7
213     6
241     5
280     5
194     5
278     5
132     5
245     5
282     5
195     5
238     4
141     4
Name: count, dtype: int64

In [6]:
# Count TP53 mutations annotated as hotspot versus non-hotspot

tp53_hotspot_counts = (
    tp53_mutations_df["hotspot"]
    .fillna("Unknown")
    .value_counts()
)

tp53_hotspot_counts

hotspot
Y    261
N     86
Name: count, dtype: int64

In [7]:
# Calculate the percentage of TP53 mutations annotated as hotspots

tp53_hotspot_summary = (
    tp53_mutations_df["hotspot"]
    .fillna("Unknown")
    .value_counts()
    .rename_axis("hotspot_status")
    .reset_index(name="count")
)

tp53_hotspot_summary["percentage"] = (
    tp53_hotspot_summary["count"]
    / tp53_hotspot_summary["count"].sum()
    * 100
).round(1)

tp53_hotspot_summary

,hotspot_status,count,percentage
0,Y,261,75.2
1,N,86,24.8


In [8]:
# Classify TP53-mutated patients by whether they carry at least one hotspot mutation

tp53_patient_hotspot = (
    tp53_mutations_df
    .groupby("submitter_id")["hotspot"]
    .apply(lambda values: "Hotspot" if (values == "Y").any() else "Non-hotspot")
)

print("Unique TP53-mutated patients:", len(tp53_patient_hotspot))

tp53_patient_hotspot.value_counts()

Unique TP53-mutated patients: 332


hotspot
Hotspot        253
Non-hotspot     79
Name: count, dtype: int64

In [9]:
# Calculate patient-level TP53 hotspot percentages

tp53_patient_hotspot_summary = (
    tp53_patient_hotspot
    .value_counts()
    .rename_axis("hotspot_status")
    .reset_index(name="patient_count")
)

tp53_patient_hotspot_summary["percentage"] = (
    tp53_patient_hotspot_summary["patient_count"]
    / tp53_patient_hotspot_summary["patient_count"].sum()
    * 100
).round(1)

tp53_patient_hotspot_summary

,hotspot_status,patient_count,percentage
0,Hotspot,253,76.2
1,Non-hotspot,79,23.8


In [10]:
# Count the number of TP53 mutation records observed for each patient

tp53_mutations_per_patient = (
    tp53_mutations_df["submitter_id"]
    .value_counts()
)

tp53_mutations_per_patient.value_counts().sort_index()

count
1    320
2     10
3      1
4      1
Name: count, dtype: int64

In [11]:
# Inspect patients with more than one TP53 mutation record

patients_with_multiple_tp53 = (
    tp53_mutations_per_patient[
        tp53_mutations_per_patient > 1
    ].index
)

multiple_tp53_df = (
    tp53_mutations_df[
        tp53_mutations_df["submitter_id"].isin(
            patients_with_multiple_tp53
        )
    ]
    [
        [
            "submitter_id",
            "tumor_aliquot_id",
            "Variant_Classification",
            "HGVSp_Short",
            "Protein_position",
            "hotspot"
        ]
    ]
    .sort_values(["submitter_id", "tumor_aliquot_id"])
)

multiple_tp53_df

,submitter_id,tumor_aliquot_id,Variant_Classification,HGVSp_Short,Protein_position,hotspot
302,TCGA-A2-A04T,TCGA-A2-A04T-01A-21W-A050-09,Missense_Mutation,p.R273H,273/393,Y
303,TCGA-A2-A04T,TCGA-A2-A04T-01A-21W-A050-09,Missense_Mutation,p.D208G,208/393,Y
301,TCGA-A7-A13D,TCGA-A7-A13D-01A-13D-A12Q-09,Nonsense_Mutation,p.E221*,221/393,N
304,TCGA-A7-A13D,TCGA-A7-A13D-01A-13D-A272-09,Nonsense_Mutation,p.E221*,221/393,N
299,TCGA-A7-A13D,TCGA-A7-A13D-01B-04D-A272-09,Nonsense_Mutation,p.E221*,221/393,N
315,TCGA-A7-A13E,TCGA-A7-A13E-01A-11D-A12Q-09,Missense_Mutation,p.R175H,175/393,Y
316,TCGA-A7-A13E,TCGA-A7-A13E-01A-11D-A12Q-09,Frame_Shift_Del,p.P77Gfs*65,76-83/393,N
65,TCGA-A7-A13E,TCGA-A7-A13E-01A-11D-A272-09,Missense_Mutation,p.R175H,175/393,Y
66,TCGA-A7-A13E,TCGA-A7-A13E-01A-11D-A272-09,Frame_Shift_Del,p.P77Gfs*65,76-83/393,N
8,TCGA-A7-A26F,TCGA-A7-A26F-01A-21D-A167-09,Missense_Mutation,p.C176F,176/393,Y
